# Wrapper Comparison: AISO vs RFE vs GA

**Goal:** Wrapper-vs-wrapper fair comparison for the reframing decision.

| Method | Category | Baseline? |
|--------|----------|-----------|
| mRMR | Filter | reference only |
| RFE (LR) | Wrapper | **main competitor** |
| GA | Wrapper / Metaheuristic | **main competitor** |
| AISO (Smart+Score) | Wrapper / Metaheuristic | our method |

Protocol identical to Exp 5: Elliptic Bitcoin, K=20, LR final eval, 5 seeds, AUC + PR-AUC.

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.feature_selection import RFE
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.cluster import AgglomerativeClustering
from sklearn.feature_selection import mutual_info_classif

np.random.seed(42)

DATA_DIR  = Path('../Elliptic Bitcoin/elliptic_bitcoin_dataset')
K_SELECT  = 20
N_TYPES   = 15
SEEDS     = [0, 7, 42, 77, 123]
GAMMA     = 0.5

print('imports ok')

## 1. Data

In [ ]:
feat_df = pd.read_csv(DATA_DIR / 'elliptic_txs_features.csv', header=None)
cls_df  = pd.read_csv(DATA_DIR / 'elliptic_txs_classes.csv')

N_FEAT = feat_df.shape[1] - 2
feat_df.columns = ['txId', 'timestep'] + [f'f{i}' for i in range(N_FEAT)]
cls_df.columns  = ['txId', 'class']

df = feat_df.merge(cls_df, on='txId')
df = df[df['class'] != 'unknown'].copy()
df['label'] = (df['class'] == '1').astype(int)

feat_cols = [f'f{i}' for i in range(N_FEAT)]
train_df  = df[df['timestep'] <= 34]
test_df   = df[df['timestep'] >  34]

X_train = train_df[feat_cols].values.astype(float)
y_train = train_df['label'].values
X_test  = test_df[feat_cols].values.astype(float)
y_test  = test_df['label'].values

print(f'Train: {X_train.shape}, illicit={y_train.sum()}')
print(f'Test:  {X_test.shape},  illicit={y_test.sum()}')

## 2. Shared Utilities

In [ ]:
def evaluate(X_tr, y_tr, X_te, y_te, feat_idx, seed=42):
    lr = LogisticRegression(class_weight='balanced', max_iter=500, random_state=seed, C=1.0)
    lr.fit(X_tr[:, feat_idx], y_tr)
    proba = lr.predict_proba(X_te[:, feat_idx])[:, 1]
    return {
        'AUC':    roc_auc_score(y_te, proba),
        'PR-AUC': average_precision_score(y_te, proba),
    }

def select_mrmr(X, y, k, seed=42):
    mi = mutual_info_classif(X, y, random_state=seed)
    C_abs = np.abs(np.corrcoef(X.T))
    np.fill_diagonal(C_abs, 0.0)
    selected  = [int(np.argmax(mi))]
    remaining = list(range(X.shape[1]))
    remaining.remove(selected[0])
    while len(selected) < k:
        scores = [mi[f] - float(np.mean(C_abs[f, selected])) for f in remaining]
        best   = remaining[int(np.argmax(scores))]
        selected.append(best); remaining.remove(best)
    return np.array(selected)

# Smart M (from Exp 5)
print('Building Smart M...')
C     = np.corrcoef(X_train.T)
C_abs = np.abs(C)
dist_matrix = 1.0 - C_abs
np.fill_diagonal(dist_matrix, 0.0)
clustering = AgglomerativeClustering(n_clusters=N_TYPES, metric='precomputed', linkage='average')
cluster_labels = clustering.fit_predict(dist_matrix)
MI_global = mutual_info_classif(X_train, y_train, random_state=42)

mean_MI = np.array([MI_global[cluster_labels == k].mean() if (cluster_labels == k).any() else 0.0 for k in range(N_TYPES)])
MI_norm = (mean_MI - mean_MI.min()) / (mean_MI.max() - mean_MI.min() + 1e-8)

M_smart = np.zeros((N_TYPES, N_TYPES))
for i in range(N_TYPES):
    for j in range(N_TYPES):
        fi = np.where(cluster_labels == i)[0]
        fj = np.where(cluster_labels == j)[0]
        if i == j:
            M_smart[i][j] = -1.0
        else:
            corr_pen = -np.mean(C_abs[np.ix_(fi, fj)])
            M_smart[i][j] = corr_pen + GAMMA * (MI_norm[j] - MI_norm[i])

print('Smart M ready')

## 3. RFE

In [ ]:
print('Running RFE...')
rfe_results = []

for seed in SEEDS:
    estimator = LogisticRegression(class_weight='balanced', max_iter=300, random_state=seed, C=1.0, solver='liblinear')
    rfe = RFE(estimator=estimator, n_features_to_select=K_SELECT, step=5)
    rfe.fit(X_train, y_train)
    feat_idx = np.where(rfe.support_)[0]
    res = evaluate(X_train, y_train, X_test, y_test, feat_idx, seed=seed)
    res['seed'] = seed
    rfe_results.append(res)
    print(f'  seed={seed}: AUC={res["AUC"]:.4f}  PR-AUC={res["PR-AUC"]:.4f}')

rfe_auc   = np.mean([r['AUC']    for r in rfe_results])
rfe_prauc = np.mean([r['PR-AUC'] for r in rfe_results])
print(f'\nRFE mean: AUC={rfe_auc:.4f}  PR-AUC={rfe_prauc:.4f}')

## 4. GA Wrapper

Chromosome: binary vector with exactly K=20 ones  
Fitness: AUC from SGD proxy on validation split  
Ops: tournament selection, uniform crossover (constrained), bit-swap mutation

In [ ]:
class GAFeatureSelector:
    def __init__(self, n_features_to_select=20, pop_size=30, n_gen=60,
                 mutation_rate=0.1, val_ratio=0.2, subsample=0.3):
        self.k            = n_features_to_select
        self.pop_size     = pop_size
        self.n_gen        = n_gen
        self.mutation_rate = mutation_rate
        self.val_ratio    = val_ratio
        self.subsample    = subsample

    def _init_pop(self, D, rng):
        pop = []
        for _ in range(self.pop_size):
            chrom = np.zeros(D, dtype=bool)
            chrom[rng.choice(D, self.k, replace=False)] = True
            pop.append(chrom)
        return pop

    def _fitness(self, chrom, X_sub, y_sub, X_val, y_val, proxy):
        idx = np.where(chrom)[0]
        try:
            proxy.fit(X_sub[:, idx], y_sub)
            proba = proxy.predict_proba(X_val[:, idx])[:, 1]
            return roc_auc_score(y_val, proba)
        except Exception:
            return 0.5

    def _crossover(self, p1, p2, rng):
        # union then randomly keep K
        union = np.where(p1 | p2)[0]
        if len(union) <= self.k:
            child = p1.copy()
            return child
        selected = rng.choice(union, self.k, replace=False)
        child = np.zeros(len(p1), dtype=bool)
        child[selected] = True
        return child

    def _mutate(self, chrom, rng):
        if rng.rand() < self.mutation_rate:
            chrom = chrom.copy()
            on  = np.where(chrom)[0]
            off = np.where(~chrom)[0]
            # swap one on ↔ one off
            chrom[rng.choice(on)]  = False
            chrom[rng.choice(off)] = True
        return chrom

    def select(self, X, y, seed=42):
        rng = np.random.RandomState(seed)
        N, D = X.shape

        val_n   = int(self.val_ratio * N)
        val_idx = rng.choice(N, val_n, replace=False)
        tr_mask = np.ones(N, bool); tr_mask[val_idx] = False
        X_val, y_val = X[val_idx], y[val_idx]
        X_t,   y_t   = X[tr_mask], y[tr_mask]

        proxy  = SGDClassifier(loss='log_loss', max_iter=5, tol=None,
                               class_weight='balanced', random_state=seed)
        sub_n  = max(50, int(self.subsample * len(X_t)))
        pop    = self._init_pop(D, rng)

        for gen in range(self.n_gen):
            sub = rng.choice(len(X_t), sub_n, replace=False)
            fits = [self._fitness(c, X_t[sub], y_t[sub], X_val, y_val, proxy) for c in pop]

            # elitism: keep top 2
            order   = np.argsort(fits)[::-1]
            new_pop = [pop[order[0]], pop[order[1]]]

            while len(new_pop) < self.pop_size:
                # tournament selection (size 3)
                t1 = rng.choice(self.pop_size, 3, replace=False)
                t2 = rng.choice(self.pop_size, 3, replace=False)
                p1 = pop[t1[np.argmax([fits[i] for i in t1])]]
                p2 = pop[t2[np.argmax([fits[i] for i in t2])]]
                child = self._crossover(p1, p2, rng)
                child = self._mutate(child, rng)
                new_pop.append(child)

            pop = new_pop
            if (gen + 1) % 10 == 0:
                print(f'    gen {gen+1:3d}  best_fitness={max(fits):.4f}')

        sub  = rng.choice(len(X_t), sub_n, replace=False)
        fits = [self._fitness(c, X_t[sub], y_t[sub], X_val, y_val, proxy) for c in pop]
        best = pop[np.argmax(fits)]
        return np.where(best)[0]

print('GA defined')

In [ ]:
print('Running GA (this takes a few minutes)...')
ga = GAFeatureSelector(n_features_to_select=K_SELECT, pop_size=30, n_gen=60, mutation_rate=0.15)
ga_results = []

for seed in SEEDS:
    print(f'  seed={seed}')
    feat_idx = ga.select(X_train, y_train, seed=seed)
    res = evaluate(X_train, y_train, X_test, y_test, feat_idx, seed=seed)
    res['seed'] = seed
    ga_results.append(res)
    print(f'  => AUC={res["AUC"]:.4f}  PR-AUC={res["PR-AUC"]:.4f}')

ga_auc   = np.mean([r['AUC']    for r in ga_results])
ga_prauc = np.mean([r['PR-AUC'] for r in ga_results])
print(f'\nGA mean: AUC={ga_auc:.4f}  PR-AUC={ga_prauc:.4f}')

## 5. AISO (Smart+Score) — rerun from Exp 5

In [ ]:
class AISOwrapper:
    def __init__(self, n_agents=20, n_iter=80, beta=0.15,
                 subsample_ratio=0.15, val_ratio=0.2,
                 T_start=1.0, T_end=0.05):
        self.n_agents        = n_agents
        self.n_iter          = n_iter
        self.beta            = beta
        self.subsample_ratio = subsample_ratio
        self.val_ratio       = val_ratio
        self.T_start         = T_start
        self.T_end           = T_end

    def select(self, X, y, cluster_labels, K_select, M, seed=42):
        rng  = np.random.RandomState(seed)
        N, D = X.shape
        K    = M.shape[0]
        nb   = min(3, self.n_agents - 1)

        val_n   = int(self.val_ratio * N)
        val_idx = rng.choice(N, val_n, replace=False)
        tr_mask = np.ones(N, bool); tr_mask[val_idx] = False
        X_val, y_val = X[val_idx],  y[val_idx]
        X_t,   y_t   = X[tr_mask],  y[tr_mask]

        proxy  = SGDClassifier(loss='log_loss', max_iter=5, tol=None,
                               class_weight='balanced', random_state=seed)
        cache  = {}
        sub_n  = max(50, int(self.subsample_ratio * len(X_t)))

        def get_mask(Wi, t, explore=True):
            if explore:
                ratio  = t / max(1, self.n_iter - 1)
                T      = self.T_start * ((self.T_end / self.T_start) ** ratio)
                logits = Wi[cluster_labels] / T
                logits -= logits.max()
                probs  = np.exp(logits); probs /= probs.sum()
                return tuple(rng.choice(D, K_select, replace=False, p=probs))
            else:
                return tuple(np.argsort(Wi[cluster_labels])[-K_select:])

        def get_score(mask_key, X_sub, y_sub):
            if mask_key in cache: return cache[mask_key]
            feat = list(mask_key)
            try:
                proxy.fit(X_sub[:, feat], y_sub)
                proba = proxy.predict_proba(X_val[:, feat])[:, 1]
                s = roc_auc_score(y_val, proba)
            except Exception:
                s = 0.5
            cache[mask_key] = s
            return s

        W      = rng.dirichlet(np.ones(K), self.n_agents)
        scores = np.zeros(self.n_agents)
        for i in range(self.n_agents):
            sub = rng.choice(len(X_t), sub_n, replace=False)
            scores[i] = get_score(get_mask(W[i], 0), X_t[sub], y_t[sub])

        for t in range(self.n_iter):
            Cm = W @ M @ W.T
            np.fill_diagonal(Cm, 0.0)
            sub = rng.choice(len(X_t), sub_n, replace=False)
            X_sub, y_sub = X_t[sub], y_t[sub]
            for i in range(self.n_agents):
                att = np.argsort(Cm[i])[-nb:]
                att_scores = np.array([get_score(get_mask(W[j], t), X_sub, y_sub) for j in att])
                max_s = att_scores.max() + 1e-8
                bj    = att[np.argmax(Cm[i, att] * att_scores / max_s)]
                W_new = (1 - self.beta) * W[i] + self.beta * W[bj]
                W[i]  = W_new / W_new.sum()
                scores[i] = get_score(get_mask(W[i], t), X_sub, y_sub)

        final_lr = LogisticRegression(solver='liblinear', C=1.0, max_iter=500,
                                      class_weight='balanced', random_state=seed)
        fseen, final_scores = {}, []
        for i in range(self.n_agents):
            mk = get_mask(W[i], self.n_iter, explore=False)
            if mk not in fseen:
                final_lr.fit(X_t[:, list(mk)], y_t)
                proba = final_lr.predict_proba(X_val[:, list(mk)])[:, 1]
                fseen[mk] = roc_auc_score(y_val, proba)
            final_scores.append(fseen[mk])
        return np.array(list(get_mask(W[np.argmax(final_scores)], self.n_iter, explore=False)))

print('Running AISO (Smart+Score)...')
wrapper = AISOwrapper(n_agents=20, n_iter=80, beta=0.15)
aiso_results = []

for seed in SEEDS:
    feat_idx = wrapper.select(X_train, y_train, cluster_labels, K_SELECT, M_smart, seed=seed)
    res = evaluate(X_train, y_train, X_test, y_test, feat_idx, seed=seed)
    res['seed'] = seed
    aiso_results.append(res)
    print(f'  seed={seed}: AUC={res["AUC"]:.4f}  PR-AUC={res["PR-AUC"]:.4f}')

aiso_auc   = np.mean([r['AUC']    for r in aiso_results])
aiso_prauc = np.mean([r['PR-AUC'] for r in aiso_results])
print(f'\nAISO mean: AUC={aiso_auc:.4f}  PR-AUC={aiso_prauc:.4f}')

## 6. Results

In [ ]:
# mRMR reference (deterministic)
mrmr_idx = select_mrmr(X_train, y_train, K_SELECT)
mrmr_res = evaluate(X_train, y_train, X_test, y_test, mrmr_idx)

summary = {
    'mRMR (filter, ref)':    {'AUC': mrmr_res['AUC'],  'PR-AUC': mrmr_res['PR-AUC'],  'std_auc': 0, 'std_pr': 0, 'cat': 'Filter'},
    'RFE (wrapper)':         {'AUC': rfe_auc,           'PR-AUC': rfe_prauc,            'std_auc': np.std([r['AUC'] for r in rfe_results]),  'std_pr': np.std([r['PR-AUC'] for r in rfe_results]),  'cat': 'Wrapper'},
    'GA (wrapper)':          {'AUC': ga_auc,            'PR-AUC': ga_prauc,             'std_auc': np.std([r['AUC'] for r in ga_results]),   'std_pr': np.std([r['PR-AUC'] for r in ga_results]),   'cat': 'Wrapper'},
    'AISO Smart (wrapper)':  {'AUC': aiso_auc,          'PR-AUC': aiso_prauc,           'std_auc': np.std([r['AUC'] for r in aiso_results]), 'std_pr': np.std([r['PR-AUC'] for r in aiso_results]), 'cat': 'Wrapper'},
}

print(f'\n{"Method":<26} {"Cat":<10} {"AUC":>8} {"±":>6} {"PR-AUC":>8} {"±":>6}')
print('-' * 70)
for name, s in summary.items():
    print(f'  {name:<24} {s["cat"]:<10} {s["AUC"]:>8.4f} {s["std_auc"]:>6.4f} {s["PR-AUC"]:>8.4f} {s["std_pr"]:>6.4f}')

print()
print('=== DECISION GATE ===')
beats_rfe = aiso_prauc > rfe_prauc
beats_ga  = aiso_prauc > ga_prauc
print(f'AISO vs RFE  (PR-AUC): {aiso_prauc:.4f} vs {rfe_prauc:.4f}  -> {"WIN" if beats_rfe else "LOSE"}')
print(f'AISO vs GA   (PR-AUC): {aiso_prauc:.4f} vs {ga_prauc:.4f}  -> {"WIN" if beats_ga  else "LOSE"}')
if beats_rfe and beats_ga:
    print('=> Reframe viable')
elif beats_rfe or beats_ga:
    print('=> Partial — negotiate scope')
else:
    print('=> Keep current framing')

In [ ]:
methods = list(summary.keys())
colors  = ['#95a5a6', '#3498db', '#e67e22', '#2ecc71']
x       = np.arange(len(methods))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, metric, title in zip(axes, ['AUC', 'PR-AUC'], ['AUC', 'PR-AUC']):
    vals = [summary[m][metric]     for m in methods]
    errs = [summary[m]['std_auc' if metric == 'AUC' else 'std_pr'] for m in methods]
    bars = ax.bar(x, vals, yerr=errs, capsize=6, color=colors, alpha=0.85, width=0.6)
    ax.set_xticks(x)
    ax.set_xticklabels([m.replace(' (', '\n(') for m in methods], fontsize=9)
    ax.set_title(f'{title} — Wrapper Comparison', fontsize=11)
    ax.set_ylabel(title)
    ax.grid(alpha=0.3, axis='y')
    ymin = min(vals) - 0.03
    ymax = max(v + e for v, e in zip(vals, errs)) + 0.03
    ax.set_ylim(ymin, ymax)
    for xi, (v, e) in enumerate(zip(vals, errs)):
        ax.text(xi, v + e + 0.005, f'{v:.4f}', ha='center', fontsize=8)

plt.suptitle(f'Feature Selection: AISO vs Wrappers (K={K_SELECT}, Elliptic Bitcoin, 5 seeds)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('wrapper_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('saved wrapper_comparison.png')